# DGGAL HEALPix polyline linetrace

Step-by-step visualization of polyline → DGGAL cells, matching vgrid [`polyline2dggal`](https://github.com/opengeoshub/vgrid/blob/main/vgrid/conversion/vector2dggs/vector2dggal.py). Same animation style as [`01_h3_linetrace.ipynb`](01_h3_linetrace.ipynb) / [`04_rhealpix_linetrace.ipynb`](04_rhealpix_linetrace.ipynb).

For each line part:

1. Remove duplicate and collinear vertices
2. For each segment: `latlon2dggal` at start and end
3. If the start cell already covers the endpoint → next segment
4. Otherwise, for each current cell:
   - show **all neighbors**
   - next frame: **unchosen** (non-intersecting) neighbors disappear
   - keep intersecting neighbors and continue until the endpoint

This notebook uses **DGGAL HEALPix** at **resolution 16**.

Input: [`multipolyline.geojson`](https://raw.githubusercontent.com/opengeoshub/vopendata/main/shape/multipolyline.geojson) — **5 features**, **6 line parts** (disjoint branches + one closed loop). All rows/parts are loaded and drawn in distinct colors.

## Install necessary packages

In [ ]:
%pip install vgrid geopandas matplotlib imageio pillow
# optional for MP4:
%pip install imageio-ffmpeg

In [ ]:
"""Step-by-step polyline2dggal neighbor-walk animation (multipolyline.geojson)."""
from collections import deque
from pathlib import Path

import geopandas as gpd
import imageio.v2 as imageio
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon as MplPolygon
from shapely.geometry import LineString, MultiLineString
from shapely.geometry import Point as ShapelyPoint

from vgrid.conversion.dggs2geo.dggal2geo import dggal2geo
from vgrid.conversion.latlon2dggs import latlon2dggal
from vgrid.conversion.vector2dggs.vector2dggal import (
    _strip_duplicate_and_collinear_vertices,
    polyline2dggal,
)
from vgrid.utils.constants import DGGAL_TYPES
from dggal import *

app = Application(appGlobals=globals())
pydggal_setup(app)

URL = "https://raw.githubusercontent.com/opengeoshub/vopendata/main/shape/polyline2.geojson"
DGGS_TYPE = "healpix"
RESOLUTION = 18
SPLIT_ANTIMERIDIAN = False
OUT_GIF = "polyline2dggal.gif"
OUT_MP4 = "polyline2dggal.mp4"
DPI = 120
PART_COLORS = ["#1f4e79", "#c55a11", "#2e7d32", "#b71c1c", "#6a1b9a", "#4e342e"]


def cell_patches(cell_polys, facecolor, edgecolor, alpha=0.55, lw=0.4):
    patches = []
    for poly in cell_polys:
        if poly is None or poly.is_empty:
            continue
        patches.append(MplPolygon(list(poly.exterior.coords), closed=True))
    return PatchCollection(
        patches, facecolor=facecolor, edgecolor=edgecolor, alpha=alpha, linewidths=lw
    )


def polylines_from_feature(feature):
    if feature.geom_type == "LineString":
        return [feature]
    if feature.geom_type == "MultiLineString":
        return list(feature.geoms)
    return []


def polylines_from_gdf(gdf):
    parts = []
    for geom in gdf.geometry:
        if geom is None or geom.is_empty:
            continue
        parts.extend(polylines_from_feature(geom))
    return parts


def feature_from_gdf(gdf):
    parts = polylines_from_gdf(gdf)
    if not parts:
        raise ValueError("No line geometries found in input GeoJSON")
    if len(parts) == 1:
        return parts[0]
    return MultiLineString(parts)


def part_color(part_index):
    return PART_COLORS[part_index % len(PART_COLORS)]


def cell_poly(zone_id):
    return dggal2geo(DGGS_TYPE, zone_id, split_antimeridian=SPLIT_ANTIMERIDIAN)


def render_frame(
    parts,
    segment_line,
    path_polys,
    title,
    path,
    resolution,
    current_poly=None,
    endpoint_polys=None,
    neighbor_polys=None,
    chosen_polys=None,
    waypoints=None,
    active_part=None,
):
    fig, ax = plt.subplots(figsize=(8, 8))
    minx, miny, maxx, maxy = MultiLineString(parts).bounds
    pad = max(maxx - minx, maxy - miny) * 0.08 or 0.01
    ax.set_xlim(minx - pad, maxx + pad)
    ax.set_ylim(miny - pad, maxy + pad)

    for i, line in enumerate(parts):
        color = part_color(i)
        lw = 3.5 if active_part == i else 2.0
        alpha = 1.0 if active_part is None or active_part == i else 0.45
        gpd.GeoSeries([line]).plot(
            ax=ax, facecolor="none", edgecolor=color, lw=lw, alpha=alpha
        )
    if segment_line is not None:
        gpd.GeoSeries([segment_line]).plot(
            ax=ax, facecolor="none", edgecolor="#1f77b4", lw=3.5
        )
    if waypoints:
        lons, lats = zip(*waypoints)
        ax.scatter(lons, lats, c="#d62728", s=30, zorder=5)

    visited = list(path_polys) if path_polys else []
    if current_poly is not None:
        visited = [
            p
            for p in visited
            if p is not current_poly and not p.equals(current_poly)
        ]
    if visited:
        ax.add_collection(cell_patches(visited, "#2ca02c", "#1a5f1a", alpha=0.45))

    if neighbor_polys:
        ax.add_collection(
            cell_patches(neighbor_polys, "#9e9e9e", "#616161", alpha=0.4)
        )

    if chosen_polys:
        ax.add_collection(
            cell_patches(chosen_polys, "#f58518", "#e65100", alpha=0.7, lw=1.2)
        )

    if endpoint_polys:
        ax.add_collection(
            cell_patches(endpoint_polys, "#d62728", "#8b0000", alpha=0.7)
        )

    if current_poly is not None:
        ax.add_collection(
            cell_patches([current_poly], "#ffcc00", "#cc8800", alpha=0.9, lw=2.5)
        )

    ax.plot([], [], color="#ffcc00", lw=4, label="current cell")
    ax.plot([], [], color="#2ca02c", lw=4, label="path so far")
    ax.plot([], [], color="#9e9e9e", lw=4, label="all neighbors")
    ax.plot([], [], color="#f58518", lw=4, label="chosen neighbors")
    ax.plot([], [], color="#d62728", lw=4, label="segment endpoints")
    ax.legend(loc="upper right", fontsize=8)
    ax.text(
        0.02,
        0.98,
        f"DGGAL {DGGS_TYPE} resolution: {resolution}",
        transform=ax.transAxes,
        fontsize=9,
        va="top",
        ha="left",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.9),
        zorder=6,
    )
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.25)
    fig.subplots_adjust(left=0.08, right=0.92, top=0.92, bottom=0.08)
    fig.savefig(path, dpi=DPI, facecolor="white")
    plt.close(fig)


def polyline2dggal_with_frames(parts, feature, resolution, frame_dir):
    """Mirror polyline2dggal with frame capture for all parts."""
    frame_dir.mkdir(parents=True, exist_ok=True)
    frames = []
    idx = 0

    if not parts:
        return frames, []

    dggs_class_name = DGGAL_TYPES[DGGS_TYPE]["class_name"]
    dggrs = globals()[dggs_class_name]()
    ordered_cells = []
    path_polys = []

    def snap(
        segment_line,
        title,
        current=None,
        endpoints=None,
        neighbors=None,
        chosen=None,
        active_part=None,
        waypoints=None,
    ):
        nonlocal idx
        p = frame_dir / f"frame_{idx:04d}.png"
        render_frame(
            parts,
            segment_line,
            path_polys,
            title,
            p,
            resolution,
            current_poly=current,
            endpoint_polys=endpoints,
            neighbor_polys=neighbors,
            chosen_polys=chosen,
            waypoints=waypoints,
            active_part=active_part,
        )
        frames.append(p)
        idx += 1

    def add_path_cell(cell_id, poly=None):
        if ordered_cells and ordered_cells[-1] == cell_id:
            return
        ordered_cells.append(cell_id)
        if poly is None:
            poly = cell_poly(cell_id)
        if poly is not None and not poly.is_empty:
            path_polys.append(poly)

    n_parts = len(parts)
    n_verts = sum(len(list(line.coords)) for line in parts)
    n_closed = sum(1 for line in parts if line.coords[0] == line.coords[-1])
    snap(
        None,
        f"1. Input res {resolution} ({n_parts} parts, {n_verts} vertices, "
        f"{n_closed} closed loop{'s' if n_closed != 1 else ''})",
    )
    snap(None, "2. All parts (distinct colors)")

    for part_i, polyline in enumerate(parts, start=1):
        raw_n = len(list(polyline.coords))
        coords = _strip_duplicate_and_collinear_vertices(polyline)
        if len(coords) < 2:
            continue

        part_cells_start = len(ordered_cells)
        is_closed = coords[0] == coords[-1]
        loop_note = " [closed loop]" if is_closed else ""
        snap(
            None,
            f"3. Part {part_i}/{n_parts}{loop_note}: cleaned "
            f"{raw_n} → {len(coords)} vertices",
            active_part=part_i - 1,
            waypoints=coords,
        )

        for seg_i in range(len(coords) - 1):
            start_xy = coords[seg_i]
            end_xy = coords[seg_i + 1]
            segment_line = LineString([start_xy, end_xy])

            start_id = latlon2dggal(
                DGGS_TYPE, start_xy[1], start_xy[0], resolution
            )
            end_id = latlon2dggal(DGGS_TYPE, end_xy[1], end_xy[0], resolution)
            start_poly = cell_poly(start_id)
            end_poly = cell_poly(end_id)
            end_pt = ShapelyPoint(end_xy[0], end_xy[1])
            n_seg = len(coords) - 1

            snap(
                segment_line,
                f"3b. Part {part_i} seg {seg_i + 1}/{n_seg}: "
                f"{start_id} → {end_id}",
                endpoints=[start_poly, end_poly],
                active_part=part_i - 1,
            )

            if segment_line.is_empty or segment_line.length == 0:
                add_path_cell(start_id, start_poly)
                snap(
                    segment_line,
                    f"4. Part {part_i} seg {seg_i + 1}: degenerate segment",
                    current=start_poly,
                    endpoints=[start_poly, end_poly],
                    active_part=part_i - 1,
                )
                continue

            if start_poly is None:
                add_path_cell(start_id)
                if end_id != start_id:
                    add_path_cell(end_id, end_poly)
                continue

            if start_id == end_id or start_poly.covers(end_pt):
                add_path_cell(start_id, start_poly)
                if end_id != start_id:
                    add_path_cell(end_id, end_poly)
                snap(
                    segment_line,
                    f"4. Part {part_i} seg {seg_i + 1}: start covers endpoint",
                    current=start_poly,
                    endpoints=[start_poly, end_poly],
                    active_part=part_i - 1,
                )
                continue

            visited = set()
            queue = deque([start_id])
            reached_endpoint = False
            walk_step = 0

            while queue:
                cell_id = queue.popleft()
                if cell_id in visited:
                    continue
                visited.add(cell_id)

                cell_polygon = cell_poly(cell_id)
                if cell_polygon is None or cell_polygon.is_empty:
                    continue
                if cell_id != start_id and not cell_polygon.intersects(
                    segment_line
                ):
                    continue

                add_path_cell(cell_id, cell_polygon)
                walk_step += 1

                if cell_id == end_id or cell_polygon.covers(end_pt):
                    reached_endpoint = True
                    snap(
                        segment_line,
                        f"4. Part {part_i} seg {seg_i + 1} step {walk_step}: "
                        f"{cell_id} reached endpoint ({len(ordered_cells)} total)",
                        current=cell_polygon,
                        endpoints=[start_poly, end_poly],
                        active_part=part_i - 1,
                    )
                    continue

                zone = dggrs.getZoneFromTextID(cell_id)
                neighbors = dggrs.getZoneNeighbors(zone)
                all_neighbor_polys = []
                chosen_neighbor_polys = []

                for neighbor in neighbors:
                    neighbor_id = dggrs.getZoneTextID(neighbor)
                    neighbor_polygon = cell_poly(neighbor_id)
                    if neighbor_polygon is None or neighbor_polygon.is_empty:
                        continue
                    all_neighbor_polys.append(neighbor_polygon)
                    if neighbor_polygon.intersects(segment_line):
                        chosen_neighbor_polys.append(neighbor_polygon)
                        if neighbor_id not in visited:
                            queue.append(neighbor_id)

                snap(
                    segment_line,
                    f"4. Part {part_i} seg {seg_i + 1} step {walk_step}: "
                    f"{cell_id} all neighbors ({len(all_neighbor_polys)})",
                    current=cell_polygon,
                    endpoints=[start_poly, end_poly],
                    neighbors=all_neighbor_polys,
                    active_part=part_i - 1,
                )
                snap(
                    segment_line,
                    f"4. Part {part_i} seg {seg_i + 1} step {walk_step}: "
                    f"{cell_id} chosen neighbors ({len(chosen_neighbor_polys)})",
                    current=cell_polygon,
                    endpoints=[start_poly, end_poly],
                    chosen=chosen_neighbor_polys,
                    active_part=part_i - 1,
                )

            if not reached_endpoint and end_id not in visited:
                add_path_cell(end_id, end_poly)
                snap(
                    segment_line,
                    f"4. Part {part_i} seg {seg_i + 1}: appended endpoint {end_id}",
                    current=end_poly,
                    endpoints=[start_poly, end_poly],
                    active_part=part_i - 1,
                )

        snap(
            None,
            f"5. Part {part_i} done ({len(ordered_cells) - part_cells_start} cells)",
            active_part=part_i - 1,
        )

    snap(None, f"6. Complete path ({len(ordered_cells)} cells)")

    rows = polyline2dggal(
        DGGS_TYPE,
        feature,
        resolution,
        split_antimeridian=SPLIT_ANTIMERIDIAN,
    )
    from_polyline2dggal = [row[f"dggal_{DGGS_TYPE}"] for row in rows]
    if set(from_polyline2dggal) != set(ordered_cells):
        print(
            "Warning: cell set differs from polyline2dggal:",
            len(ordered_cells),
            "vs",
            len(from_polyline2dggal),
        )
    elif from_polyline2dggal != ordered_cells:
        print(
            "Note: same unique cells as polyline2dggal, different order "
            f"({len(ordered_cells)} vs {len(from_polyline2dggal)} rows)"
        )

    return frames, ordered_cells


def main():
    gdf = gpd.read_file(URL)
    parts = polylines_from_gdf(gdf)
    feature = feature_from_gdf(gdf)
    print(f"Loaded {len(gdf)} feature(s), {len(parts)} line part(s)")

    frame_dir = Path("_polyline2dggal_frames")
    frames, cell_ids = polyline2dggal_with_frames(
        parts, feature, RESOLUTION, frame_dir
    )
    imageio.mimsave(OUT_GIF, [imageio.imread(f) for f in frames], duration=0.9)
    print(f"Wrote {OUT_GIF} ({len(frames)} frames, {len(cell_ids)} final cells)")
    try:
        writer = imageio.get_writer(OUT_MP4, fps=1.2)
        for f in frames:
            writer.append_data(imageio.imread(f))
        writer.close()
        print(f"Wrote {OUT_MP4}")
    except Exception as e:
        print(f"MP4 skipped ({e}). GIF is enough.")


if __name__ == "__main__":
    main()


Loaded 1 feature(s), 1 line part(s)
Wrote polyline2dggal.gif (125 frames, 59 final cells)
MP4 skipped (Could not find a backend to open `polyline2dggal.mp4`` with iomode `w?`.
Based on the extension, the following plugins might add capable backends:
  FFMPEG:  pip install imageio[ffmpeg]
  pyav:  pip install imageio[pyav]). GIF is enough.
